## Performance Tuning with `ZORDER` and `Liquid Clustering`

### Create the Sample Dataset

In [0]:
from pyspark.sql.functions import *
import random

data = []

cities = ["London", "Manchester", "Birmingham", "Leeds"]
segments = ["Retail", "Premium", "Business"]

for i in range(100000):
    data.append((
        f"CUST-{i}",
        random.choice(cities),
        random.choice(segments),
        random.randint(100, 10000),
        random.randint(1, 10)
    ))

df = spark.createDataFrame(data, [
    "customer_id", "city", "segment", "revenue", "transactions"
])

display(df)

### Save Table Partitioned by City

In [0]:
df.write \
    .format("delta") \
    .option("maxRecordsPerFile", 500) \
    .partitionBy("city") \
    .mode("overwrite") \
    .saveAsTable("default.sales_ordering_demo")

### Describe the Managed Table

In [0]:
%sql
DESCRIBE DETAIL default.sales_ordering_demo;

### Run Query Before Z-Ordering (Slow Performance)

In [0]:
%sql
SELECT city, SUM(revenue)
FROM default.sales_ordering_demo
WHERE city = 'London' AND segment = 'Retail'
GROUP BY city;

### Z-Order by Customer Segment

In [0]:
%sql
OPTIMIZE default.sales_ordering_demo
ZORDER BY (segment);

### Describe Managed Table After Z-Ordering

In [0]:
%sql
DESCRIBE DETAIL default.sales_ordering_demo;

### Run Query After Z-Ordering (Optimal Performance)

In [0]:
%sql
SELECT city, SUM(revenue)
FROM default.sales_ordering_demo
WHERE city = 'London' AND segment = 'Retail'
GROUP BY city;

### Create a Table with Liquid Clustering

In [0]:
%sql

CREATE TABLE default.customer_sales_lc
USING DELTA
CLUSTER BY (city, segment)
AS 
SELECT 
  customer_id,
  city,
  segment,
  revenue,
  transactions
FROM default.sales_ordering_demo;

### Describe the Clustered Managed Table

In [0]:
%sql
DESCRIBE DETAIL default.customer_sales_lc;